# 06 - LLM-Based Personalised Re-ranking

Uses embeddings to shortlist cases and an OpenAI language model to re-rank results according to a user profile, generate relevance explanations, and produce a research-oriented predicted outcome.

**Expected input:** `data/processed/preprocessed_wto_cases.csv`  
**Security:** requires `OPENAI_API_KEY` in a local `.env` file; never place a key directly in this notebook.

> Install dependencies once from the repository root with `pip install -r requirements.txt`. Run the notebooks in numerical order.

In [ ]:
from pathlib import Path

# Resolve repository paths whether Jupyter starts in the repository root or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_PDF_DIR = DATA_DIR / "raw" / "wto_case_pdfs"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

RAW_PDF_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = PROCESSED_DIR / "wto_cases.csv"
PROCESSED_DATA_PATH = PROCESSED_DIR / "preprocessed_wto_cases.csv"

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load OPENAI_API_KEY from a local .env file or the operating-system environment.
# Never commit the .env file or a real API key to GitHub.
load_dotenv(PROJECT_ROOT / ".env")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Copy .env.example to .env and add your own key."
    )

client = OpenAI()


In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI()  # Reads OPENAI_API_KEY from the environment

# Load the preprocessed data
df = pd.read_csv(PROCESSED_DATA_PATH)

# Use cleaned and preprocessed columns 
df['Title'] = df['Title'].fillna('').astype(str)
df['Articles_lemmatized'] = df['Articles_lemmatized'].fillna('').astype(str)
df['Summary_lemmatized'] = df['Summary_lemmatized'].fillna('').astype(str)

# Create combined embedding input
df['embedding_input'] = (
    df['Title'] + ' ' +
    df['Articles_lemmatized'] + ' ' +
    df['Summary_lemmatized']
)

# Define the query and user profile
user_query = "US - China Case Laws Using Article VIII of GATT"
user_profile = """
User is an international trade lawyer. Interested in GATT Article VIII and US-China disputes.
Jurisdiction Focus: United States and China. Explanation Level: International Trade Law.
"""

# Embedding function
def get_embedding(text, model="text-embedding-3-small"):
    response = client.embeddings.create(input=[text], model=model)
    return response.data[0].embedding

# Generate embeddings for all cases
print("Generating embeddings...")
case_embeddings = [
    get_embedding(text[:1000])  
    for text in df['embedding_input']
]

# Generate embedding for query + user profile
query_embedding = get_embedding(user_query + " " + user_profile)

# Compute similarity
print("Computing similarity scores...")
similarities = cosine_similarity([query_embedding], case_embeddings)[0]

# Select top 30 most relevant cases
top_30_indices = np.argsort(similarities)[::-1][:30]
top_30_cases = df.iloc[top_30_indices][['Case ID', 'Title']].reset_index(drop=True)

# Display and save top 30 ranked Case IDs and Titles
print("\n Top 30 Most Relevant Cases:")
for i, row in top_30_cases.iterrows():
    print(f"{i+1}. {row['Case ID']} – {row['Title']}")
top_30_cases.to_csv(RESULTS_DIR / "top_30_query_1.csv", index=False)


Generating embeddings...
Computing similarity scores...

 Top 30 Most Relevant Cases:
1. ds437 – US – COUNTERVAILING MEASURES (CHINA)1
2. ds471 – US – ANTI-DUMPING METHODOLOGIES (CHINA)1
3. ds437A – US – COUNTERVAILING MEASURES (CHINA) (ARTICLE 21.5 – CHINA)1
4. ds395 – CHINA – RAW MATERIALS1
5. ds165 – US – CERTAIN EC PRODUCTS1
6. ds322A – US – ZEROING (JAPAN) (ARTICLE 21.5 – JAPAN)1
7. ds440 – CHINA – AUTOS (US)1
8. ds460 – CHINA – HP-SSST (JAPAN/EUROPEAN UNION)1
9. ds258 – US – STEEL SAFEGUAR
10. ds285A – US – GAMBLING (ARTICLE 21.5 – ANTIGUA AND BARBUDA)1
11. ds492 – EU – POULTRY MEAT (CHINA)1
12. ds344 – US – STAINLESS STEEL (MEXICO)1
13. ds152 – US – SECTION 301 TRADE ACT1
14. ds322 – US – ZEROING (JAPAN)1
15. ds510 – US – RENEWABLE ENERGY (INDIA)1
16. ds432 – CHINA – RARE EARTHS1
17. ds381A – US – TUNA II (ARTICLE 21.5 – MEXICO)1
18. ds339 – CHINA – AUTO PARTS1
19. ds257A – US – SOFTWOOD LUMBER IV (ARTICLE 21.5 – CANADA)1
20. ds316B – (ARTICLE 21.5 – US)1
21. ds384 – US – COOL (

In [4]:
import json
import time
import math
from openai import OpenAI
import pandas as pd 

# Initialize OpenAI client
client = OpenAI()  # Reads OPENAI_API_KEY from the environment

# Top 30 cases from embeddings (Case IDs + Title)
df = pd.read_csv(RESULTS_DIR / "top_30_query_1.csv")

# Define user profile
user_profile = {
    "name": "Madiha Ahmadi",
    "role": "International Trade Lawyer",
    "interests": ["GATT Article VIII", "US-China disputes"],
    "jurisdiction_focus": ["United States", "China"],
    "explanation_level": "International Trade Law"
}

# Define the query
query = "US - China Case Laws using Article VIII of GATT"

# Define chunking
chunk_size = 10
num_chunks = math.ceil(len(df) / chunk_size)

# Loop through chunks and rank 5 cases from each with GPT-4
for i in range(num_chunks):
    print(f"\n Processing chunk {i + 1}...")

    chunk = df[i * chunk_size:(i + 1) * chunk_size]
    chunk_dict = chunk.to_dict(orient="records")
    chunk_text = json.dumps(chunk_dict, indent=2)

# Build prompt
    prompt = f"""
You are a legal assistant helping an international trade lawyer.

## User Profile:
- Name: {user_profile['name']}
- Role: {user_profile['role']}
- Interests: {', '.join(user_profile['interests'])}
- Jurisdictional Focus: {', '.join(user_profile['jurisdiction_focus'])}
- Explanation Level: {user_profile['explanation_level']}

## Query:
"{query}"

## Task:
From the list below, select the **5 most relevant cases** based on legal reasoning and relevance to the query.
Return only a JSON list in this format:
[
  {{
    "id": "dsXXX",
    "title": "Full Case Title"
  }},
  ...
]

## Case Laws List:
{chunk_text}
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4",
            temperature=0.0,
            messages=[
                {"role": "system", "content": "You are a legal assistant helping an international trade lawyer."},
                {"role": "user", "content": prompt}
            ]
        )

        result = json.loads(response.choices[0].message.content)
        print(f"✅ Got {len(result)} cases from chunk {i + 1}:")
        for case in result:
            print(f"   • {case['id']} – {case['title']}")
            

        time.sleep(5)

    except Exception as e:
        print(f"❌ Error in chunk {i + 1}: {e}") 
        


 Processing chunk 1...
✅ Got 5 cases from chunk 1:
   • ds437 – US – COUNTERVAILING MEASURES (CHINA)1
   • ds471 – US – ANTI-DUMPING METHODOLOGIES (CHINA)1
   • ds437A – US – COUNTERVAILING MEASURES (CHINA) (ARTICLE 21.5 – CHINA)1
   • ds395 – CHINA – RAW MATERIALS1
   • ds440 – CHINA – AUTOS (US)1

 Processing chunk 2...
✅ Got 5 cases from chunk 2:
   • ds492 – EU – POULTRY MEAT (CHINA)1
   • ds152 – US – SECTION 301 TRADE ACT1
   • ds432 – CHINA – RARE EARTHS1
   • ds339 – CHINA – AUTO PARTS1
   • ds316B – (ARTICLE 21.5 – US)1

 Processing chunk 3...
✅ Got 5 cases from chunk 3:
   • ds384 – US – COOL (ARTICLE 21.5 – CANADA AND MEXICO)1
   • ds108A – US – FSC (ARTICLE 21.5 – EC)1
   • ds294 – US – ZEROING (EC)1
   • ds194 – US – EXPORT RESTRAINTS1
   • ds464 – US – WASHING MACHINES1


In [6]:
import json
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI()  # Reads OPENAI_API_KEY from the environment

# 15 cases from LLM chunk ranking
top_15_cases = [
    {"id": "ds437", "title": "US – COUNTERVAILING MEASURES (CHINA)1"},
    {"id": "ds471", "title": "US – ANTI-DUMPING METHODOLOGIES (CHINA)1"},
    {"id": "ds437A", "title": "US – COUNTERVAILING MEASURES (CHINA) (ARTICLE 21.5 – CHINA)1"},
    {"id": "ds395", "title": "CHINA – RAW MATERIALS1"},
    {"id": "ds440", "title": "CHINA – AUTOS (US)1"},
    {"id": "ds492", "title": "EU - POULTRY MEAT (CHINA)1"},
    {"id": "ds152", "title": "US – SECTION 301 TRADE ACT1"},
    {"id": "ds432", "title": "CHINA – RARE EARTHS1"},
    {"id": "ds339", "title": "CHINA – AUTO PARTS1"},
    {"id": "ds316B", "title": "(ARTICLE 21.5 – US)1"},
    {"id": "ds384", "title": "US - Cool (ARTICLE 21.5 - CANADA AND MEXICO)1"},
    {"id": "ds108A", "title": "US - FSC (ARTICLE 21.5 - EC)1"},
    {"id": "ds294", "title": "US – ZEROING (EC)1"},
    {"id": "ds194", "title": "US - EXPORT RESTRAINTS1"},
    {"id": "ds464", "title": "US – WASHING MACHINES1"},
]

# Define user profile
user_profile = {
    "name": "Madiha Ahmadi",
    "role": "International Trade Lawyer",
    "interests": ["GATT Article VIII", "US-China disputes"],
    "jurisdiction_focus": ["United States", "China"],
    "explanation_level": "International Trade Law"
}

# Define the query
query = "US - China Case Laws using Article VIII of GATT"

# Build prompt
prompt = f"""
You are a legal assistant helping an international trade lawyer.

## User Profile:
- Name: {user_profile['name']}
- Role: {user_profile['role']}
- Interests: {', '.join(user_profile['interests'])}
- Jurisdictional Focus: {', '.join(user_profile['jurisdiction_focus'])}
- Explanation Level: {user_profile['explanation_level']}

## Query:
"{query}"

## Task:
From the list of 15 case laws below, select the **5 most relevant** to this query. 
Provide a **ranked list** from most to least relevant based on legal context, query and user interests.

Return format:
1. Case ID – Case Title


# Case List (JSON):
{json.dumps(top_15_cases, indent=2)}
"""

# Call GPT-4
response = client.chat.completions.create(
    model="gpt-4",
    temperature=0.0,
    messages=[
        {"role": "system", "content": "You are a legal assistant helping an international trade lawyer."},
        {"role": "user", "content": prompt}
    ]
)

# Print output
print("\n Top 5 Cases:\n")
print(response.choices[0].message.content)



 Top 5 Cases:

1. Case ID – ds437 – US – COUNTERVAILING MEASURES (CHINA)1
2. Case ID – ds471 – US – ANTI-DUMPING METHODOLOGIES (CHINA)1
3. Case ID – ds437A – US – COUNTERVAILING MEASURES (CHINA) (ARTICLE 21.5 – CHINA)1
4. Case ID – ds152 – US – SECTION 301 TRADE ACT1
5. Case ID – ds440 – CHINA – AUTOS (US)1


In [7]:
import pandas as pd 

# Top 5 Case IDs
top_5_case_ids = ["ds437", "ds471", "ds437A", "ds152", "ds440"]

# Load the dataset
df = pd.read_csv(PROCESSED_DATA_PATH)

# Filter and preserve the original ranking order
df_filtered = df[df["Case ID"].isin(top_5_case_ids)]
df_ordered = df_filtered.set_index("Case ID").loc[top_5_case_ids].reset_index()
top_5_data = df_ordered[["Case ID", "Title", "Summary_lemmatized"]].to_dict(orient="records")

# Format for prompt
cases_text = "\n".join([
    f"{i+1}. {case['Case ID']} – {case['Title']}: {case['Summary_lemmatized']}" for i, case in enumerate(top_5_data)
])


In [10]:
from openai import OpenAI

client = OpenAI()  # Reads OPENAI_API_KEY from the environment

# Define user profile and query
user_profile = {
    "name": "Madiha Ahmadi",
    "role": "International Trade Lawyer",
    "interests": ["GATT Article VIII", "US-China disputes"],
    "jurisdiction_focus": ["United States", "China"],
    "explanation_level": "International Trade Law"
}
query = "US - China Case Laws using Article VIII of GATT"

# Build prompt
prompt = f"""
You are a legal assistant helping an international trade lawyer.

## User Profile:
- Name: {user_profile['name']}
- Role: {user_profile['role']}
- Interests: {', '.join(user_profile['interests'])}
- Jurisdictional Focus: {', '.join(user_profile['jurisdiction_focus'])}
- Explanation Level: {user_profile['explanation_level']}

## Query:
"{query}"

Below are 5 WTO case law summaries retrieved for this query:

{cases_text}

## Task:
1. For each of the 5 selected cases, provide a **professional legal justification** for its relevance, written for an {user_profile['explanation_level']} explanation level.
2. Predict what kind of **ruling** a WTO panel might make in a similar dispute based on these precedents.


Return your output as:
List of Top 5 Case Laws (Case ID - Title):
Justifications for Each Selected Case:
General Predicted Outcome:
"""

# Send to GPT-4
response = client.chat.completions.create(
    model="gpt-4",
    temperature=0.0,
    max_tokens=2500,
    messages=[
        {"role": "system", "content": "You are a legal assistant helping an international trade lawyer."},
        {"role": "user", "content": prompt}
    ]
)

# Print result
print(response.choices[0].message.content)


List of Top 5 Case Laws (Case ID - Title):
1. ds437 – US – COUNTERVAILING MEASURES (CHINA)
2. ds471 – US – ANTI-DUMPING METHODOLOGIES (CHINA)
3. ds437A – US – COUNTERVAILING MEASURES (CHINA) (ARTICLE 21.5 – CHINA)
4. ds152 – US – SECTION 301 TRADE ACT
5. ds440 – CHINA – AUTOS (US)

Justifications for Each Selected Case:
1. ds437 – US – COUNTERVAILING MEASURES (CHINA): This case is relevant as it deals with the determination of public bodies and the use of in-country prices as benchmarks for benefit analysis. It provides insight into how the WTO views the role of state-owned enterprises and the use of domestic prices in determining benefits.
2. ds471 – US – ANTI-DUMPING METHODOLOGIES (CHINA): This case is significant as it addresses the use of different anti-dumping methodologies by the US. It provides a precedent on how the WTO views the use of different price comparison methodologies and the requirement to consider evidence of non-target prices.
3. ds437A – US – COUNTERVAILING MEASURE

In [11]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI()  # Reads OPENAI_API_KEY from the environment

# Load the preprocessed data
df = pd.read_csv(PROCESSED_DATA_PATH)

# Use cleaned and preprocessed columns 
df['Title'] = df['Title'].fillna('').astype(str)
df['Articles_lemmatized'] = df['Articles_lemmatized'].fillna('').astype(str)
df['Summary_lemmatized'] = df['Summary_lemmatized'].fillna('').astype(str)

# Create combined embedding input
df['embedding_input'] = (
    df['Title'] + ' ' +
    df['Articles_lemmatized'] + ' ' +
    df['Summary_lemmatized']
)

# Define query and user profile
user_query = "Case Laws involving European Communities as Complainant and United States as Respondent"
user_profile = """
User is an academic. Interested in European Communities and United States disputes as complainant and respondent.
Jurisdiction Focus: European Communities and United States. Explanation Level: International Trade Interests.
"""

# Embedding function
def get_embedding(text, model="text-embedding-3-small"):
    response = client.embeddings.create(input=[text], model=model)
    return response.data[0].embedding

# Generate embeddings for all cases
print("Generating embeddings...")
case_embeddings = [
    get_embedding(text[:1000])  
    for text in df['embedding_input']
]

# Generate embedding for query + user profile
query_embedding = get_embedding(user_query + " " + user_profile)

# Compute similarity
print("Computing similarity scores...")
similarities = cosine_similarity([query_embedding], case_embeddings)[0]

# Select top 30 most relevant cases
top_30_indices = np.argsort(similarities)[::-1][:30]
top_30_cases = df.iloc[top_30_indices][['Case ID', 'Title']].reset_index(drop=True)

# Display and save top 30 ranked Case IDs and Titles
print("\n Top 30 Most Relevant Cases:")
for i, row in top_30_cases.iterrows():
    print(f"{i+1}. {row['Case ID']} – {row['Title']}")
top_30_cases.to_csv(RESULTS_DIR / "top_30_query_2.csv", index=False)


Generating embeddings...
Computing similarity scores...

 Top 30 Most Relevant Cases:
1. ds353A – US – LARGE CIVIL AIRCRAFT (2ND COMPLAINT) (ARTICLE 21.5 – EU)1
2. ds165 – US – CERTAIN EC PRODUCTS1
3. ds62 – EC – COMPUTER EQUIPMENT1
4. ds437 – US – COUNTERVAILING MEASURES (CHINA)1
5. ds285A – US – GAMBLING (ARTICLE 21.5 – ANTIGUA AND BARBUDA)1
6. ds322A – US – ZEROING (JAPAN) (ARTICLE 21.5 – JAPAN)1
7. ds577 – US — RIPE OLIVES FROM SPAIN1
8. ds437A – US – COUNTERVAILING MEASURES (CHINA) (ARTICLE 21.5 – CHINA)1
9. ds257A – US – SOFTWOOD LUMBER IV (ARTICLE 21.5 – CANADA)1
10. ds108A – US – FSC (ARTICLE 21.5 – EC)1
11. ds353 – US – LARGE CIVIL AIRCRAFT (2ND COMPLAINT)1
12. ds108B – US – FSC (ARTICLE 21.5 – EC II)1
13. ds316B – (ARTICLE 21.5 – US)1
14. ds231 – EC – SARDINES1
15. ds27B – EC – BANANAS III (ARTICLE 21.5 – US)1
16. ds277A – US – SOFTWOOD LUMBER VI (ARTICLE 21.5 – CANADA)1
17. ds510 – US – RENEWABLE ENERGY (INDIA)1
18. ds322 – US – ZEROING (JAPAN)1
19. ds294 – US – ZEROING (EC)

In [12]:
import json
import time
import math
from openai import OpenAI
import pandas as pd 

# Initialize OpenAI client
client = OpenAI()  # Reads OPENAI_API_KEY from the environment

# Top 30 cases from embeddings (Case IDs + Title)
df = pd.read_csv(RESULTS_DIR / "top_30_query_2.csv")

# Define user profile
user_profile = {
    "name": "Madiha Ahmadi",
    "role": "Academic",
    "interests": ["EC-US disputes", "Complainant", "Respondent"],
    "jurisdiction_focus": ["United States", "European Communities"],
    "explanation_level": "International Trade Interests"
}

# Define the query
query = "Case Laws involving European Communities as Complainant and United States as Respondent" 

# Define chunking
chunk_size = 10
num_chunks = math.ceil(len(df) / chunk_size)

# Loop through chunks and rank 5 cases from each with GPT-4
for i in range(num_chunks):
    print(f"\n Processing chunk {i + 1}...")

    chunk = df[i * chunk_size:(i + 1) * chunk_size]
    chunk_dict = chunk.to_dict(orient="records")
    chunk_text = json.dumps(chunk_dict, indent=2)

# Build prompt
    prompt = f"""
You are a Ph.D. student helping a law academic professor.

## User Profile:
- Name: {user_profile['name']}
- Role: {user_profile['role']}
- Interests: {', '.join(user_profile['interests'])}
- Jurisdictional Focus: {', '.join(user_profile['jurisdiction_focus'])}
- Explanation Level: {user_profile['explanation_level']}

## Query:
"{query}"

## Task:
From the list below, select the **5 most relevant cases** based on legal reasoning and relevance to the query.
Return only a JSON list in this format:
[
  {{
    "id": "dsXXX",
    "title": "Full Case Title"
  }},
  ...]

## Case Laws List:
{chunk_text}
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4",
            temperature=0.0,
            messages=[
                {"role": "system", "content": "You are a Ph.D. student helping a law academic professor."},
                {"role": "user", "content": prompt}
            ]
        )

        result = json.loads(response.choices[0].message.content)
        print(f"✅ Got {len(result)} cases from chunk {i + 1}:")
        for case in result:
            print(f"   • {case['id']} – {case['title']}")
            

        time.sleep(5)

    except Exception as e:
        print(f"❌ Error in chunk {i + 1}: {e}")
        


 Processing chunk 1...
✅ Got 5 cases from chunk 1:
   • ds353A – US – LARGE CIVIL AIRCRAFT (2ND COMPLAINT) (ARTICLE 21.5 – EU)1
   • ds165 – US – CERTAIN EC PRODUCTS1
   • ds108A – US – FSC (ARTICLE 21.5 – EC)1
   • ds577 – US — RIPE OLIVES FROM SPAIN1
   • ds62 – EC – COMPUTER EQUIPMENT1

 Processing chunk 2...
✅ Got 5 cases from chunk 2:
   • ds353 – US – LARGE CIVIL AIRCRAFT (2ND COMPLAINT)1
   • ds108B – US – FSC (ARTICLE 21.5 – EC II)1
   • ds27B – EC – BANANAS III (ARTICLE 21.5 – US)1
   • ds294 – US – ZEROING (EC)1
   • ds316B – (ARTICLE 21.5 – US)1

 Processing chunk 3...
✅ Got 5 cases from chunk 3:
   • ds69 – EC – POULTRY1
   • ds194 – US – EXPORT RESTRAINTS1
   • ds258 – US – STEEL SAFEGUAR
   • ds277 – US – SOFTWOOD LUMBER VI1
   • ds234 – US – OFFSET ACT (BYRD AMENDMENT)1


In [13]:
import json
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI()  # Reads OPENAI_API_KEY from the environment

# 15 cases from LLM chunk ranking
top_15_cases = [
    {"id": "ds353A", "title": "US – LARGE CIVIL AIRCRAFT (2ND COMPLAINT) (ARTICLE 21.5 - EU)1"},
    {"id": "ds165", "title": "US – CERTAIN EC PRODUCTS1"},
    {"id": "ds108A", "title": "US - FSC (ARTICLE 21.5 - EC)1"},
    {"id": "ds577", "title": "US – OLIVES FROM SPAIN1"},
    {"id": "ds62", "title": "EC - COMPUTER EQUIPMENT1"},
    {"id": "ds353", "title": "US – LARGE CIVIL AIRCRAFT (2ND COMPLAINT)1"},
    {"id": "ds108B", "title": "US - FSC (ARTICLE 21.5 - EC II)1"},
    {"id": "ds27B", "title": "EC - BANANAS III (ARTICLE 21.5 - US)1"},
    {"id": "ds294", "title": "US - ZEROING (EC)1"},
    {"id": "ds316B", "title": "(ARTICLE 21.5 – US)1"},
    {"id": "ds69", "title": "EC – POULTRY1"},
    {"id": "ds194", "title": "US - EXPORT RESTRAINTS1"},
    {"id": "ds258", "title": "US – STEEL SAFEGUARDS"},
    {"id": "ds277", "title": "US – SOFTWOOD LUMBER VI1"},  
    {"id": "ds234", "title": "US – OFFSET ACT (BYRD AMENDMENT)1"},
]

# Define user profile
user_profile = {
    "name": "Madiha Ahmadi",
    "role": "Academic",
    "interests": ["EC-US disputes", "Complainant", "Respondent"],
    "jurisdiction_focus": ["United States", "European Communities"],
    "explanation_level": "International Trade Interests"
}

# Define query
query = "Case Laws involving European Communities as Complainant and United States as Respondent" 

# Build prompt
prompt = f"""
You are a Ph.D. student helping a law academic professor. 

## User Profile:
- Name: {user_profile['name']}
- Role: {user_profile['role']}
- Interests: {', '.join(user_profile['interests'])}
- Jurisdictional Focus: {', '.join(user_profile['jurisdiction_focus'])}
- Explanation Level: {user_profile['explanation_level']}

## Query:
"{query}"

## Task:
From the list of 15 case laws below, select the **5 most relevant** to this query. 
Provide a **ranked list** from most to least relevant based on legal context, query and user interests.

Return format:
1. Case ID – Case Title

# Case List (JSON):
{json.dumps(top_15_cases, indent=2)}
"""

# Call GPT-4
response = client.chat.completions.create(
    model="gpt-4",
    temperature=0.0,
    messages=[
        {"role": "system", "content": "You are a Ph.D. student helping a law academic professor."},
        {"role": "user", "content": prompt}
    ]
)

# Print output
print("\nTop 5 Cases:\n")
print(response.choices[0].message.content)



Top 5 Cases:

1. Case ID: ds353A – Case Title: US – LARGE CIVIL AIRCRAFT (2ND COMPLAINT) (ARTICLE 21.5 - EU)1
2. Case ID: ds108A – Case Title: US - FSC (ARTICLE 21.5 - EC)1
3. Case ID: ds108B – Case Title: US - FSC (ARTICLE 21.5 - EC II)1
4. Case ID: ds294 – Case Title: US - ZEROING (EC)1
5. Case ID: ds165 – Case Title: US – CERTAIN EC PRODUCTS1


In [34]:
import pandas as pd 

# Top 5 Case IDs
top_5_case_ids = ["ds353A", "ds108A", "ds108B", "ds294", "ds165"]

# Load the dataset
df = pd.read_csv(PROCESSED_DATA_PATH)

# Filter and preserve the original ranking order
df_filtered = df[df["Case ID"].isin(top_5_case_ids)]
df_ordered = df_filtered.set_index("Case ID").loc[top_5_case_ids].reset_index()
top_5_data = df_ordered[["Case ID", "Title", "Summary_lemmatized"]].to_dict(orient="records")

# Format for prompt
cases_text = "\n".join([
    f"{i+1}. {case['Case ID']} – {case['Title']}: {case['Summary_lemmatized']}" for i, case in enumerate(top_5_data)
])


In [35]:
from openai import OpenAI

client = OpenAI()  # Reads OPENAI_API_KEY from the environment

# Define user profile and query
user_profile = {
    "name": "Madiha Ahmadi",
    "role": "Academic",
    "interests": ["EC - US disputes", "Complainant", "Respondent"],
    "jurisdiction_focus": ["United States", "European Communities"],
    "explanation_level": "International Trade Interests"
}
query = "Case Laws involving European Communities as Complainant and United States as Respondent"

# Build prompt
prompt = f"""
You are a Ph.D. student helping a law academic professor. 

## User Profile:
- Name: {user_profile['name']}
- Role: {user_profile['role']}
- Interests: {', '.join(user_profile['interests'])}
- Jurisdictional Focus: {', '.join(user_profile['jurisdiction_focus'])}
- Explanation Level: {user_profile['explanation_level']}

## Query:
"{query}"

Below are 5 WTO case law summaries retrieved for this query:

{cases_text} 

## Task:
1. For each of the 5 selected cases, provide a **professional legal justification** for its relevance, written for an {user_profile['explanation_level']} explanation level.
2. Predict what kind of **ruling** a WTO panel might make in a similar dispute based on these precedents.


Return your output as:
List of Top 5 Case Laws (Case ID - Title)
Justifications for Each Selected Case
- General Predicted Outcome
"""

# Send to GPT-4
response = client.chat.completions.create(
    model="gpt-4",
    temperature=0.0,
    max_tokens=2500,
    messages=[
        {"role": "system", "content": "You are a Ph.D. student helping a law academic professor."},
        {"role": "user", "content": prompt}
    ]
)

# Print result
print(response.choices[0].message.content)


List of Top 5 Case Laws (Case ID - Title)
1. ds353A – US – LARGE CIVIL AIRCRAFT (2ND COMPLAINT) (ARTICLE 21.5 – EU)
2. ds108A – US – FSC (ARTICLE 21.5 – EC)
3. ds108B – US – FSC (ARTICLE 21.5 – EC II)
4. ds294 – US – ZEROING (EC)
5. ds165 – US – CERTAIN EC PRODUCTS

Justifications for Each Selected Case:

1. ds353A – US – LARGE CIVIL AIRCRAFT (2ND COMPLAINT) (ARTICLE 21.5 – EU): This case is relevant as it involves a dispute over subsidies provided by the US to Boeing, a large civil aircraft manufacturer. The European Communities (EC) argued that these subsidies were inconsistent with the Agreement on Subsidies and Countervailing Measures (ASCM). The case provides insights into how the WTO interprets and applies the ASCM, particularly in relation to the definition of a subsidy, the assessment of benefits conferred, and the determination of adverse effects.

2. ds108A – US – FSC (ARTICLE 21.5 – EC): This case involves a dispute over the US's Foreign Sales Corporation (FSC) scheme, which

In [18]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI()  # Reads OPENAI_API_KEY from the environment

# Load the preprocessed data
df = pd.read_csv(PROCESSED_DATA_PATH)

# Use cleaned and preprocessed columns 
df['Title'] = df['Title'].fillna('').astype(str)
df['Articles_lemmatized'] = df['Articles_lemmatized'].fillna('').astype(str)
df['Summary_lemmatized'] = df['Summary_lemmatized'].fillna('').astype(str)

# Create combined embedding input
df['embedding_input'] = (
    df['Title'] + ' ' +
    df['Articles_lemmatized'] + ' ' +
    df['Summary_lemmatized']
)

# Define query and user profile
user_query = "Case Laws involving Australia with ASCM Article Application"
user_profile = """
User is a policy maker. Interested in ASCM Articles, legal reasoning, and compliance to WTO rulings.
Jurisdiction Focus: Australia. Explanation Level: Policies.
"""

# Embedding function
def get_embedding(text, model="text-embedding-3-small"):
    response = client.embeddings.create(input=[text], model=model)
    return response.data[0].embedding

# Generate embeddings for all cases
print("Generating embeddings...")
case_embeddings = [
    get_embedding(text[:1000])  
    for text in df['embedding_input']
]

# Generate embedding for query + user profile
query_embedding = get_embedding(user_query + " " + user_profile)

# Compute similarity
print("Computing similarity scores...")
similarities = cosine_similarity([query_embedding], case_embeddings)[0]

# Select top 30 most relevant cases
top_30_indices = np.argsort(similarities)[::-1][:30]
top_30_cases = df.iloc[top_30_indices][['Case ID', 'Title']].reset_index(drop=True)

# Display and save top 30 ranked Case IDs and Titles
print("\n Top 30 Most Relevant Cases:")
for i, row in top_30_cases.iterrows():
    print(f"{i+1}. {row['Case ID']} – {row['Title']}")
top_30_cases.to_csv(RESULTS_DIR / "top_30_query_3.csv", index=False)


Generating embeddings...
Computing similarity scores...

 Top 30 Most Relevant Cases:
1. ds18A – AUSTRALIA – SALMON (ARTICLE 21.5 – CANADA)1
2. ds126 – AUSTRALIA – AUTOMOTIVE LEATHER II1
3. ds126A – AUSTRALIA – AUTOMOTIVE LEATHER II (ARTICLE 21.5 – US)1
4. ds367 – AUSTRALIA – APPLES1
5. ds18 – AUSTRALIA – SALMON1
6. ds437A – US – COUNTERVAILING MEASURES (CHINA) (ARTICLE 21.5 – CHINA)1
7. ds437 – US – COUNTERVAILING MEASURES (CHINA)1
8. ds257A – US – SOFTWOOD LUMBER IV (ARTICLE 21.5 – CANADA)1
9. ds316B – (ARTICLE 21.5 – US)1
10. ds529 – AUSTRALIA – ANTI-DUMPING MEASURES ON PAPER1
11. ds322A – US – ZEROING (JAPAN) (ARTICLE 21.5 – JAPAN)1
12. ds277A – US – SOFTWOOD LUMBER VI (ARTICLE 21.5 – CANADA)1
13. ds234 – US – OFFSET ACT (BYRD AMENDMENT)1
14. ds108B – US – FSC (ARTICLE 21.5 – EC II)1
15. ds285A – US – GAMBLING (ARTICLE 21.5 – ANTIGUA AND BARBUDA)1
16. ds353A – US – LARGE CIVIL AIRCRAFT (2ND COMPLAINT) (ARTICLE 21.5 – EU)1
17. ds108A – US – FSC (ARTICLE 21.5 – EC)1
18. ds436 – US – 

In [19]:
import json
import time
import math
from openai import OpenAI
import pandas as pd 

# Initialize OpenAI client
client = OpenAI()  # Reads OPENAI_API_KEY from the environment

# Top 30 cases from embeddings (Case IDs + Title)
df = pd.read_csv(RESULTS_DIR / "top_30_query_3.csv")

# Define the user profile
user_profile = {
    "name": "Madiha Ahmadi",
    "role": "Policy Maker",
    "interests": ["ASCM Articles", "Legal reasoning", "Compliance to WTo rulings"],
    "jurisdiction_focus": ["Australia"],
    "explanation_level": "Policies"
}

# Define query
query = "Case Laws involving Australia with ASCM Article Application" 

# Define chunking
chunk_size = 10
num_chunks = math.ceil(len(df) / chunk_size)

# Loop through chunks and rank 5 cases from each with GPT-4
for i in range(num_chunks):
    print(f"\n Processing chunk {i + 1}...")

    chunk = df[i * chunk_size:(i + 1) * chunk_size]
    chunk_dict = chunk.to_dict(orient="records")
    chunk_text = json.dumps(chunk_dict, indent=2)

# Build prompt
    prompt = f"""
You are a policy making assistant helping a senior policy maker.
    
## User Profile:
- Name: {user_profile['name']}
- Role: {user_profile['role']}
- Interests: {', '.join(user_profile['interests'])}
- Jurisdictional Focus: {', '.join(user_profile['jurisdiction_focus'])}
- Explanation Level: {user_profile['explanation_level']}

## Query:
"{query}"

## Task:
From the list below, select the **5 most relevant cases** based on legal reasoning and relevance to the query.
Return only a JSON list in this format:
[
  {{
    "id": "dsXXX",
    "title": "Full Case Title"
  }},
  ...]

## Case Laws List:
{chunk_text}
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4",
            temperature=0.0,
            messages=[
                {"role": "system", "content": "You are a policy making assistant helping a senior policy maker."},
                {"role": "user", "content": prompt}
            ]
        )

        result = json.loads(response.choices[0].message.content)
        print(f"✅ Got {len(result)} cases from chunk {i + 1}:")
        for case in result:
            print(f"   • {case['id']} – {case['title']}")
            

        time.sleep(5)

    except Exception as e:
        print(f"❌ Error in chunk {i + 1}: {e}") 
        


 Processing chunk 1...
✅ Got 5 cases from chunk 1:
   • ds18A – AUSTRALIA – SALMON (ARTICLE 21.5 – CANADA)1
   • ds126 – AUSTRALIA – AUTOMOTIVE LEATHER II1
   • ds126A – AUSTRALIA – AUTOMOTIVE LEATHER II (ARTICLE 21.5 – US)1
   • ds367 – AUSTRALIA – APPLES1
   • ds529 – AUSTRALIA – ANTI-DUMPING MEASURES ON PAPER1

 Processing chunk 2...
❌ Error in chunk 2: Expecting value: line 1 column 1 (char 0)

 Processing chunk 3...
✅ Got 5 cases from chunk 3:
   • ds436A – US – CARBON STEEL (INDIA)1
   • ds62 – EC – COMPUTER EQUIPMENT1
   • ds135 – EC – ASBESTOS1
   • ds456 – INDIA – SOLAR CELLS 1
   • ds165 – US – CERTAIN EC PRODUCTS1


In [22]:
import json
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI()  # Reads OPENAI_API_KEY from the environment

# 10 cases from LLM chunk ranking
top_15_cases = [
    {"id": "ds18A", "title": "AUSTRALIA - SALMON (ARTICLE 21.5 - CANADA)1"},
    {"id": "ds126", "title": "AUSTRALIA - AUTOMOTIVE LEATHER II1"},
    {"id": "ds126A", "title": "AUTOMOTIVE LEATHER II (ARTICLE 21.5 – US)1"},
    {"id": "ds367", "title": "AUSTRALIA – APPLES1"},
    {"id": "ds529", "title": "AUSTRALIA – ANTI-DUMPING MEASURES ON PAPER1"},
    {"id": "ds436A", "title": "US – CARBON STEEL (INDIA)1"},
    {"id": "ds62", "title": "EC – COMPUTER EQUIPMENT1"},
    {"id": "ds135", "title": "EC – ASBESTOS1"},
    {"id": "ds456", "title": "INDIA – SOLAR CELLS1"},
    {"id": "ds165", "title": "US – CERTAIN EC PRODUCTS1"},
]

# Define the user profile
user_profile = {
    "name": "Madiha Ahmadi",
    "role": "Policy Maker",
    "interests": ["ASCM Articles", "Legal reasoning", "Compliance to WTo rulings"],
    "jurisdiction_focus": ["Australia"],
    "explanation_level": "Policies"
}

# Define query
query = "Case Laws involving Australia with ASCM Article Application" 

# Build prompt
prompt = f"""
You are a policy making assistant helping a senior policy maker.

## User Profile:
- Name: {user_profile['name']}
- Role: {user_profile['role']}
- Interests: {', '.join(user_profile['interests'])}
- Jurisdictional Focus: {', '.join(user_profile['jurisdiction_focus'])}
- Explanation Level: {user_profile['explanation_level']}

## Query:
"{query}"

## Task:
From the list of 10 case laws below, select the **5 most relevant** to this query. 
Provide a **ranked list** from most to least relevant based on legal context, query and user interests.

Return format:
1. Case ID – Case Title

# Case List (JSON):
{json.dumps(top_15_cases, indent=2)}
"""

# Call GPT-4
response = client.chat.completions.create(
    model="gpt-4",
    temperature=0.0,
    messages=[
        {"role": "system", "content": "You are a policy making assistant helping a senior policy maker."},
               {"role": "user", "content": prompt}
    ]
)

# Print output
print("\nTop 5 Cases:\n")
print(response.choices[0].message.content)



Top 5 Cases:

Based on the query and user interests, the 5 most relevant case laws involving Australia with ASCM Article Application are:

1. Case ID – ds126 - AUSTRALIA - AUTOMOTIVE LEATHER II1
2. Case ID – ds126A - AUTOMOTIVE LEATHER II (ARTICLE 21.5 – US)1
3. Case ID – ds18A - AUSTRALIA - SALMON (ARTICLE 21.5 - CANADA)1
4. Case ID – ds367 - AUSTRALIA – APPLES1
5. Case ID – ds529 - AUSTRALIA – ANTI-DUMPING MEASURES ON PAPER1

The cases were selected based on their relevance to Australia and the potential application of ASCM articles. The ranking is based on the specificity of the case title to the query and user interests.


In [23]:
import pandas as pd 

# Top 5 Case IDs
top_5_case_ids = ["ds126", "ds126A", "ds18A", "ds367", "ds529"]

# Load the dataset
df = pd.read_csv(PROCESSED_DATA_PATH)

# Filter and preserve the original ranking order
df_filtered = df[df["Case ID"].isin(top_5_case_ids)]
df_ordered = df_filtered.set_index("Case ID").loc[top_5_case_ids].reset_index()
top_5_data = df_ordered[["Case ID", "Title", "Summary_lemmatized"]].to_dict(orient="records")

# Format for prompt
cases_text = "\n".join([
    f"{i+1}. {case['Case ID']} – {case['Title']}: {case['Summary_lemmatized']}" for i, case in enumerate(top_5_data)
])


In [26]:
from openai import OpenAI

client = OpenAI()  # Reads OPENAI_API_KEY from the environment

# Define user profile and query
user_profile = {
    "name": "Madiha Ahmadi",
    "role": "Policy Maker",
    "interests": ["ASCM Articles", "Legal reasoning", "Compliance to WTo rulings"],
    "jurisdiction_focus": ["Australia"],
    "explanation_level": "Policies"
}
query = "Case Laws involving Australia with ASCM Article Application"

# Build prompt
prompt = f"""
You are a policy making assistant helping a senior policy maker.

## User Profile:
- Name: {user_profile['name']}
- Role: {user_profile['role']}
- Interests: {', '.join(user_profile['interests'])}
- Jurisdictional Focus: {', '.join(user_profile['jurisdiction_focus'])}
- Explanation Level: {user_profile['explanation_level']}

## Query:
"{query}"

Below are 5 WTO case law summaries retrieved for this query:

{cases_text} 

## Task:
1. For each of the 5 selected cases, provide a **professional legal justification** for its relevance, written for an {user_profile['explanation_level']} explanation level.
2. Predict what kind of **ruling** a WTO panel might make in a similar dispute based on these precedents.


Return your output as:
List of Top 5 Case Laws (Case ID - Title)
Justifications for Each Selected Case
General Predicted Outcome
"""

# Send to GPT-4
response = client.chat.completions.create(
    model="gpt-4",
    temperature=0.0,
    max_tokens=2500,
    messages=[
        {"role": "system", "content": "You are a policy making assistant helping a senior policy maker."},
        {"role": "user", "content": prompt}
    ]
)

# Print result
print(response.choices[0].message.content)


List of Top 5 Case Laws:
1. ds126 – AUSTRALIA – AUTOMOTIVE LEATHER II
2. ds126A – AUSTRALIA – AUTOMOTIVE LEATHER II (ARTICLE 21.5 – US)
3. ds18A – AUSTRALIA – SALMON (ARTICLE 21.5 – CANADA)
4. ds367 – AUSTRALIA – APPLES
5. ds529 – AUSTRALIA – ANTI-DUMPING MEASURES ON PAPER

Justifications for Each Selected Case:

1. ds126 – AUSTRALIA – AUTOMOTIVE LEATHER II: This case is relevant as it involves the application of ASCM Article 3.1(a) which prohibits subsidies contingent upon export performance. The panel's interpretation of the term "subsidy" and its application to the facts of the case provides valuable insight into how such provisions may be applied in future disputes.

2. ds126A – AUSTRALIA – AUTOMOTIVE LEATHER II (ARTICLE 21.5 – US): This case is a follow-up to the previous one and provides further clarification on the interpretation of ASCM Article 4.7, specifically the requirement to withdraw prohibited subsidies. The panel's finding that repayment in full of the prohibited subsid

In [3]:
import numpy as np

# Step 1: Define Ground Truth relevant cases
ground_truth = {
    "ds437", "ds437A", "ds517", "ds440", "ds395"  
}

# Step 2: Define LLM ranked top-5 Case IDs (in order)
predicted_ranking = [
    "ds437",  # 1
    "ds471",  # 2
    "ds437A", # 3
    "ds152",  # 4
    "ds440",  # 5
]

# Evaluation metrics 
def precision(predicted, relevant):
    hits = sum(1 for pid in predicted if pid in relevant)
    return hits / len(predicted)

def recall(predicted, relevant):
    hits = sum(1 for pid in predicted if pid in relevant)
    return hits / len(relevant)

def dcg(predicted, relevant):
    return sum((1 / np.log2(i + 2)) if pid in relevant else 0 for i, pid in enumerate(predicted))

def idcg(relevant, k):
    ideal_hits = min(len(relevant), k)
    return sum(1 / np.log2(i + 2) for i in range(ideal_hits))

def ndcg(predicted, relevant):
    return dcg(predicted, relevant) / idcg(relevant, len(predicted))

# Run evaluation 
P = precision(predicted_ranking, ground_truth)
R = recall(predicted_ranking, ground_truth)
N = ndcg(predicted_ranking, ground_truth)

# Output 
print(" LLM Ranking Evaluation (Full Top-5):")
print(f" Precision: {P:.2f}")
print(f" Recall:    {R:.2f}")
print(f" nDCG:      {N:.2f}")


 LLM Ranking Evaluation (Full Top-5):
 Precision: 0.60
 Recall:    0.60
 nDCG:      0.64


In [30]:
import numpy as np

# Step 1: Define Ground Truth relevant cases
ground_truth = {
    "ds353A", "ds108A", "ds165", "ds34", "ds108B"  
}

# Step 2: Define LLM ranked top-5 Case IDs (in order)
predicted_ranking = [
    "ds353A",  # 1
    "ds108A",  # 2
    "ds108B", # 3
    "ds294",  # 4
    "ds165",  # 5
]

# Evaluation metrics
def precision(predicted, relevant):
    hits = sum(1 for pid in predicted if pid in relevant)
    return hits / len(predicted)

def recall(predicted, relevant):
    hits = sum(1 for pid in predicted if pid in relevant)
    return hits / len(relevant)

def dcg(predicted, relevant):
    return sum((1 / np.log2(i + 2)) if pid in relevant else 0 for i, pid in enumerate(predicted))

def idcg(relevant, k):
    ideal_hits = min(len(relevant), k)
    return sum(1 / np.log2(i + 2) for i in range(ideal_hits))

def ndcg(predicted, relevant):
    return dcg(predicted, relevant) / idcg(relevant, len(predicted))

# Run evaluation
P = precision(predicted_ranking, ground_truth)
R = recall(predicted_ranking, ground_truth)
N = ndcg(predicted_ranking, ground_truth)

# Output
print(" LLM Ranking Evaluation (Full Top-5):")
print(f" Precision: {P:.2f}")
print(f" Recall:    {R:.2f}")
print(f" nDCG:      {N:.2f}")


 LLM Ranking Evaluation (Full Top-5):
 Precision: 0.80
 Recall:    0.80
 nDCG:      0.85


In [31]:
import numpy as np

# Step 1: Define Ground Truth relevant cases
ground_truth = {
    "ds126", "ds126A", "ds18A", "ds367", "ds529"  
}

# Step 2: Define LLM ranked top-5 Case IDs (in order)
predicted_ranking = [
    "ds126",  # 1
    "ds126A",  # 2
    "ds18A", # 3
    "ds367",  # 4
    "ds529",  # 5
]

# Evaluation metrics
def precision(predicted, relevant):
    hits = sum(1 for pid in predicted if pid in relevant)
    return hits / len(predicted)

def recall(predicted, relevant):
    hits = sum(1 for pid in predicted if pid in relevant)
    return hits / len(relevant)

def dcg(predicted, relevant):
    return sum((1 / np.log2(i + 2)) if pid in relevant else 0 for i, pid in enumerate(predicted))

def idcg(relevant, k):
    ideal_hits = min(len(relevant), k)
    return sum(1 / np.log2(i + 2) for i in range(ideal_hits))

def ndcg(predicted, relevant):
    return dcg(predicted, relevant) / idcg(relevant, len(predicted))

# Run evaluation
P = precision(predicted_ranking, ground_truth)
R = recall(predicted_ranking, ground_truth)
N = ndcg(predicted_ranking, ground_truth)

# Output
print(" LLM Ranking Evaluation (Full Top-5):")
print(f" Precision: {P:.2f}")
print(f" Recall:    {R:.2f}")
print(f" nDCG:      {N:.2f}")


 LLM Ranking Evaluation (Full Top-5):
 Precision: 1.00
 Recall:    1.00
 nDCG:      1.00
